# Model-based RL & world models — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def discount(rewards,gamma=0.9):
    return sum((gamma**t)*r for t,r in enumerate(rewards))
def moving_avg(x,n=5):
    x=np.asarray(x,dtype=float); return np.convolve(x,np.ones(n)/n,mode='valid')
print('setup ok')

## Learning or using a transition model to plan
Model-based RL buys sample efficiency by imagining consequences before acting.

In [ ]:
# 1) a learned transition row must be a probability distribution
counts=np.array([2,6,2],dtype=float); P=counts/counts.sum()
plt.figure(figsize=(6,3)); plt.bar(range(3),P); plt.ylim(0,1); plt.title('estimated next-state probabilities')
assert abs(P.sum()-1)<1e-12 and abs(P[1]-0.6)<1e-12

In [ ]:
# 2) belief update under partial observability
b=np.array([0.6,0.4]); P=np.array([[0.7,0.3],[0.2,0.8]]); O=np.array([0.9,0.2])
pred=b@P; post=O*pred; post=post/post.sum()
plt.figure(figsize=(6,3)); plt.bar(['hidden 0','hidden 1'],post); plt.ylim(0,1); plt.title('Bayes-filtered belief after observation')
assert abs(post[0]-0.8181818181818181)<1e-12

In [ ]:
# 3) potential-based shaping adds progress reward
phi=np.array([0.,0.4,1.0]); gamma=0.9; shaped=gamma*phi[1:]-phi[:-1]
plt.figure(figsize=(6,3)); plt.bar(['0->1','1->2'],shaped); plt.title('potential difference rewards progress')
assert np.allclose(shaped,[0.36,0.5])

In [ ]:
# 4) model rollout imagines several candidate futures
branches=np.array([[0.0,1.0,2.0],[0.5,0.5,0.5],[-0.2,0.0,3.0]])
returns=np.array([discount(b,0.9) for b in branches])
plt.figure(figsize=(6,3)); plt.bar(range(3),returns); plt.title('planning compares imagined returns')
assert int(np.argmax(returns))==0

In [ ]:
# 5) safety cost can reject a high-reward action
reward=np.array([3.0,2.4,1.5]); cost=np.array([2.0,0.8,0.1]); limit=1.0; feasible=cost<=limit
score=np.where(feasible,reward,-np.inf)
plt.figure(figsize=(6,3)); plt.bar(range(3),reward,color=['tab:red' if not f else 'tab:green' for f in feasible]); plt.axhline(0,c='k'); plt.title('best feasible action, not best raw reward')
assert int(np.argmax(score))==1